<a href="https://colab.research.google.com/github/victorm0202/temas_selectos_CD_2026/blob/main/Notebooks/1-6.%20BERT%20as%20feature%20extractor%20%2B%20pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Visual Notebook to Using BERT for the First TIme

Adapted from https://github.com/jalammar/jalammar.github.io



<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-sentence-classification.png" />

In this notebook, we will use pre-trained deep learning model to process some text. We will then use the output of that model to classify the text. The text is a list of sentences from film reviews. And we will classify each sentence as either speaking "positively" about its subject or "negatively".

## Models: Sentence Sentiment Classification
Our goal is to create a model that takes a sentence (just like the ones in our dataset) and produces either 1 (indicating the sentence carries a positive sentiment) or a 0 (indicating the sentence carries a negative sentiment). We can think of it as looking like this:

<img src="https://jalammar.github.io/images/distilBERT/sentiment-classifier-1.png" />

Under the hood, the model is actually made up of two model.

* DistilBERT processes the sentence and passes along some information it extracted from it on to the next model. DistilBERT is a smaller version of BERT developed and open sourced by the team at HuggingFace. It’s a lighter and faster version of BERT that roughly matches its performance.
* The next model, a basic Logistic Regression model from scikit learn will take in the result of DistilBERT’s processing, and classify the sentence as either positive or negative (1 or 0, respectively).

The data we pass between the two models is a vector of size 768. We can think of this of vector as an embedding for the sentence that we can use for classification.


<img src="https://jalammar.github.io/images/distilBERT/distilbert-bert-sentiment-classifier.png" />

## Dataset
The dataset we will use in this example is [SST2](https://nlp.stanford.edu/sentiment/index.html), which contains sentences from movie reviews, each labeled as either positive (has the value 1) or negative (has the value 0):


<table class="features-table">
  <tr>
    <th class="mdc-text-light-green-600">
    sentence
    </th>
    <th class="mdc-text-purple-600">
    label
    </th>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      a stirring , funny and finally transporting re imagining of beauty and the beast and 1930s horror films
    </td>
    <td class="mdc-bg-purple-50">
      1
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      apparently reassembled from the cutting room floor of any given daytime soap
    </td>
    <td class="mdc-bg-purple-50">
      0
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      they presume their audience won't sit still for a sociology lesson
    </td>
    <td class="mdc-bg-purple-50">
      0
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      this is a visually stunning rumination on love , memory , history and the war between art and commerce
    </td>
    <td class="mdc-bg-purple-50">
      1
    </td>
  </tr>
  <tr>
    <td class="mdc-bg-light-green-50" style="text-align:left">
      jonathan parker 's bartleby should have been the be all end all of the modern office anomie films
    </td>
    <td class="mdc-bg-purple-50">
      1
    </td>
  </tr>
</table>

## Installing the transformers library
Let's start by installing the huggingface transformers library so we can load our deep learning NLP model.

In [ ]:
#!pip install transformers

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import torch
import transformers as ppb
import warnings
warnings.filterwarnings('ignore')

## Importing the dataset
We'll use pandas to read the dataset and load it into a dataframe.

In [2]:
df = pd.read_csv('https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv', delimiter='\t', header=None)

For performance reasons, we'll only use 5,000 sentences from the dataset

In [3]:
batch_1 = df[:5000]

We can ask pandas how many sentences are labeled as "positive" (value 1) and how many are labeled "negative" (having the value 0)

In [4]:
batch_1[1].value_counts()

,count
1,
1,2607
0,2393


## Loading the Pre-trained BERT model
Let's now load a pre-trained BERT model.

In [5]:
# For DistilBERT:
model_class, tokenizer_class, pretrained_weights = (ppb.DistilBertModel, ppb.DistilBertTokenizer, 'distilbert-base-uncased')

## Want BERT instead of distilBERT? Uncomment the following line:
#model_class, tokenizer_class, pretrained_weights = (ppb.BertModel, ppb.BertTokenizer, 'bert-base-uncased')

# Load pretrained model/tokenizer
tokenizer = tokenizer_class.from_pretrained(pretrained_weights)
model = model_class.from_pretrained(pretrained_weights)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Right now, the variable `model` holds a pretrained distilBERT model -- a version of BERT that is smaller, but much faster and requiring a lot less memory.

## Preparing the Dataset
Before we can hand our sentences to BERT, we need to do some minimal processing to put them in the format it requires.

### Tokenization
Our first step is to tokenize the sentences -- break them up into word and subwords in the format BERT is comfortable with.

In [6]:
tokenized = batch_1[0].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tokenization-2-token-ids.png" />

### Padding
After tokenization, `tokenized` is a list of sentences -- each sentences is represented as a list of tokens. We want BERT to process our examples all at once (as one batch). It's just faster that way. For that reason, we need to pad all lists to the same size, so we can represent the input as one 2-d array, rather than a list of lists (of different lengths).

In [7]:
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)

padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])

Our dataset is now in the `padded` variable, we can view its dimensions below:

In [8]:
np.array(padded).shape

(5000, 67)

### Masking
If we directly send `padded` to BERT, that would slightly confuse it. We need to create another variable to tell it to ignore (mask) the padding we've added when it's processing its input. This is performed using the attention mask.

In order to understand what *attention mask* is, we refer to the [official documentation](https://huggingface.co/docs/transformers/glossary#attention-mask) of huggingface. But, basically, "*this argument indicates to the model which tokens should be attended to, and which should not.*", and is used for padding purposes.

![attn_mask](https://drive.google.com/uc?id=1g0sf6qzEx-osrnawYXnQ6kgTT7MSl5jT)

Figure description: For each batch, the input sequences are padded to the maximum sequence length in the batch; the attention mask is used in the model to ignore the padded areas of the input tensors (source: [medium.com](https://medium.com/@ashwinnaidu1991/text-classification-with-transformers-70acaf65c4a4))

In [10]:
sequence_a = "This is a short sequence."
sequence_b = "This is a rather long sequence. It is at least longer than the sequence A."

encoded_sequence_a = tokenizer(sequence_a)["input_ids"]
encoded_sequence_b = tokenizer(sequence_b)["input_ids"]

print('lengths of sequences:',len(encoded_sequence_a), len(encoded_sequence_b))
padded_sequences = tokenizer([sequence_a, sequence_b], padding=True)
print('Tokenized sequences after padding:\n', padded_sequences["input_ids"])
print('Attention mask:\n', padded_sequences["attention_mask"])

lengths of sequences: 8 19
Tokenized sequences after padding:
 [[101, 2023, 2003, 1037, 2460, 5537, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 2023, 2003, 1037, 2738, 2146, 5537, 1012, 2009, 2003, 2012, 2560, 2936, 2084, 1996, 5537, 1037, 1012, 102]]
Attention mask:
 [[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]


In [11]:
attention_mask = np.where(padded != 0, 1, 0)
attention_mask.shape

(5000, 67)

## And Now, Deep Learning!
Now that we have our model and inputs ready, let's run our model!

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tutorial-sentence-embedding.png" />

The `model()` function runs our sentences through BERT. The results of the processing will be returned into `last_hidden_states`.

In [ ]:
input_ids = torch.tensor(padded)
attention_mask = torch.tensor(attention_mask)

with torch.no_grad():
    last_hidden_states = model(input_ids, attention_mask=attention_mask)

Let's slice only the part of the output that we need. That is the output corresponding the first token of each sentence. The way BERT does sentence classification, is that it adds a token called `[CLS]` (for classification) at the beginning of every sentence. The output corresponding to that token can be thought of as an embedding for the entire sentence.

<img src="https://jalammar.github.io/images/distilBERT/bert-output-tensor-selection.png" />

We'll save those in the `features` variable, as they'll serve as the features to our logitics regression model.

In [ ]:
features = last_hidden_states[0][:,0,:].numpy()

In [ ]:
features.shape

(5000, 768)

The labels indicating which sentence is positive and negative now go into the `labels` variable

In [ ]:
labels = batch_1[1]

## Train/Test Split
Let's now split our datset into a training set and testing set (even though we're using 2,000 sentences from the SST2 training set).

In [ ]:
train_features, test_features, train_labels, test_labels = train_test_split(features, labels)

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-train-test-split-sentence-embedding.png" />

### [Bonus] Grid Search for Parameters
We can dive into Logistic regression directly with the Scikit Learn default parameters, but sometimes it's worth searching for the best value of the C parameter, which determines regularization strength.

In [ ]:
# parameters = {'C': np.linspace(0.0001, 100, 20)}
# grid_search = GridSearchCV(LogisticRegression(), parameters)
# grid_search.fit(train_features, train_labels)

# print('best parameters: ', grid_search.best_params_)
# print('best scrores: ', grid_search.best_score_)

We now train the LogisticRegression model. If you've chosen to do the gridsearch, you can plug the value of C into the model declaration (e.g. `LogisticRegression(C=5.2)`).

In [ ]:
lr_clf = LogisticRegression()
lr_clf.fit(train_features, train_labels)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

<img src="https://jalammar.github.io/images/distilBERT/bert-training-logistic-regression.png" />

## Evaluating Model
So how well does our model do in classifying sentences? One way is to check the accuracy against the testing dataset:

In [ ]:
lr_clf.score(test_features, test_labels)

0.8512

# Neural Network model wity pytorch

In [ ]:
from torch import nn
from torch.utils.data import DataLoader

### One class: encoder and classifier

`BERTClassifier` contains the tokenizer, pretrained encoder and a Linear → ReLU → Dropout → Linear classifier. `encode(texts)` returns the sentence vectors; `forward(texts)` also applies the classifier and returns two logits.

The encoder's weights are frozen. Its dropout remains disabled, even when `model.train()` enables training in the classifier. Embeddings are computed inside the class for each batch on each epoch, so no separate training or test feature arrays are needed.

In [ ]:
class BERTClassifier(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", max_length=128):
        super().__init__()
        self.tokenizer = ppb.AutoTokenizer.from_pretrained(model_name)
        self.encoder = ppb.AutoModel.from_pretrained(model_name)
        self.max_length = max_length
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        self.encoder.eval()

        hidden_dim = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 2),
        )

    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()
        return self

    def encode(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        inputs = self.tokenizer(
            list(texts), padding=True, truncation=True,
            max_length=self.max_length, return_tensors="pt",
        )
        device = next(self.parameters()).device
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.no_grad():
            return self.encoder(**inputs).last_hidden_state[:, 0, :]

    def forward(self, texts):
        features = self.encode(texts)
        return self.classifier(features)

### Initialize the integrated model

We use DistilBERT by default. Set `model_name="bert-base-uncased"` to use BERT instead. Both tokenization and encoder weights come from the same checkpoint. We select CUDA, MPS or CPU according to availability.

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = BERTClassifier(model_name="distilbert-base-uncased").to(device)
print("Device:", device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Device: mps
Trainable parameters: 592130


Right now, `model.encoder` holds a pretrained DistilBERT model -- a version of BERT that is smaller, but much faster and requiring a lot less memory.

Tokenization is performed now inside `model.encode(texts)`. The tokenizer splits raw text into tokens and subwords, adds special tokens such as `[CLS]` and `[SEP]`, and maps them to vocabulary IDs. `max_length=128` bounds the sequence length, including special tokens.

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-tokenization-2-token-ids.png" />

### Padding
After tokenization, each sentence is represented by a sequence of token IDs. Sentences have different lengths, but the encoder processes rectangular batches. The tokenizer uses `padding=True` to pad each batch to its longest sequence, rather than padding all 2,000 examples together.

The padded IDs are stored in `inputs["input_ids"]` inside `encode()`. Their shape is `(batch_size, sequence_length)`.

### Masking
If we directly send the padded token IDs to BERT, that would slightly confuse it. We need to create another variable to tell it to ignore (mask) the padding we've added when it's processing its input. That's what attention_mask is:

The tokenizer creates `inputs["attention_mask"]` automatically: 1 identifies a real token and 0 identifies padding. `encode()` passes both IDs and the mask to the encoder.

Now that we have our model and inputs ready, let's train our model.

Inside `encode()`, `self.encoder(**inputs)` runs each batch through BERT. Its `last_hidden_state` contains one contextual vector per token, with shape `(batch_size, sequence_length, hidden_size)`. The encoder runs under `torch.no_grad()` because its parameters are frozen.

Let's slice only the part of the output that we need. That is the output corresponding the first token of each sentence. The way BERT does sentence classification, is that it adds a token called `[CLS]` (for classification) at the beginning of every sentence. The output corresponding to that token can be thought of as an embedding for the entire sentence.

<img src="https://jalammar.github.io/images/distilBERT/bert-output-tensor-selection.png" />

`encode()` selects `last_hidden_state[:, 0, :]`, giving one vector of 768 components per sentence for the default encoder. In `forward()`, the local variable `features` receives those vectors and passes them immediately to `self.classifier`. We do not save a separate feature matrix outside the class.

The labels indicating which sentence is positive and negative are kept alongside the raw texts and returned as `labels` by each data loader

### Train/Test Split
Let's now split our datset into a training set and testing set (even though we're using 2,000 sentences from the SST2 training set).

We split raw texts and labels into 75% training and 25% testing, using a fixed seed and preserving class proportions. Data loaders return text batches; the same `model(texts)` call performs encoding and classification for either partition.

<img src="https://jalammar.github.io/images/distilBERT/bert-distilbert-train-test-split-sentence-embedding.png" />

### Preparing the PyTorch classifier

The original figure shows a split of precomputed embeddings. Here we split the texts first and compute the embeddings inside the class for each batch. The separation between training and testing is unchanged.

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    batch_1[0].tolist(), batch_1[1].tolist(), test_size=0.25, random_state=42, stratify=batch_1[1],
)

batch_size = 32
train_loader = DataLoader(list(zip(train_texts, train_labels)), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(list(zip(test_texts, test_labels)), batch_size=batch_size, shuffle=False)

### Classifier settings and training

The original example optionally searches for the regularization parameter `C` of logistic regression. For this neural classifier, the relevant settings are learning rate, number of epochs, hidden-layer size and dropout. This example uses fixed settings; use a separate validation split if you want to tune them.

We now train the classifier with cross-entropy loss and Adam. `model(texts)` handles the entire pipeline. The optimizer receives only `model.classifier.parameters()`, so the pretrained encoder remains unchanged. Cross-entropy takes the two logits directly, without applying softmax first.

In [ ]:
epochs = 10
learning_rate = 1e-3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for texts, labels in train_loader:
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(texts)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss / len(train_loader.dataset):.4f}")

Epoch 1/10 | Loss: 0.4562
Epoch 2/10 | Loss: 0.3783
Epoch 3/10 | Loss: 0.3715
Epoch 4/10 | Loss: 0.3528
Epoch 5/10 | Loss: 0.3475
Epoch 6/10 | Loss: 0.3331
Epoch 7/10 | Loss: 0.3390
Epoch 8/10 | Loss: 0.3361
Epoch 9/10 | Loss: 0.3241
Epoch 10/10 | Loss: 0.3161


In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for texts, labels in test_loader:
        predictions = model(texts).argmax(dim=1)
        correct += (predictions.cpu() == labels).sum().item()

test_accuracy = correct / len(test_loader.dataset)
print(f"Test accuracy: {test_accuracy:.3f}")

Test accuracy: 0.823


How good is this score? What can we compare it against? Let's first look at a dummy classifier:

The dummy classifier predicts the most frequent training label. We score it on the same test set as the neural network, so the two accuracies use identical examples.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(np.zeros((len(train_labels), 1)), train_labels)
baseline_accuracy = baseline.score(np.zeros((len(test_labels), 1)), test_labels)
print(f"Dummy accuracy: {baseline_accuracy:.3f}")
print(f"Neural classifier accuracy: {test_accuracy:.3f}")

Dummy accuracy: 0.522
Neural classifier accuracy: 0.823


## Classify new sentences

The model also accepts new raw texts. Softmax converts logits into class probabilities; label 0 means negative and label 1 means positive. These results use a local split of 2,000 training sentences, not the official SST-2 benchmark split.

In [ ]:
texts = ["This movie was wonderful.", "The story was boring and predictable."]
model.eval()
with torch.no_grad():
    probabilities = torch.softmax(model(texts), dim=1).cpu()

pd.DataFrame({
    "text": texts,
    "prediction": probabilities.argmax(dim=1).tolist(),
    "positive_probability": probabilities[:, 1].tolist(),
})

,text,prediction,positive_probability
0,This movie was wonderful.,1,0.999479
1,The story was boring and predictable.,0,0.007188
